# 📊 Modelado Estadístico Dinámico
**Perfil:** Cualquiera | **Entorno:** JupyterHub / K3s  
**Objetivo:** Validar la conectividad de red, descarga de datos externos y ejecución de modelos estadísticos dentro de los límites de 2 CPU / 4 GB RAM.
**Autor**: abdzher

---
> **Dataset:** UCI Water Quality (calidad de agua potable).  
> **Modelos:** Regresión Lineal Múltiple (scikit-learn) + ANOVA de un factor (scipy/statsmodels).  
> **Propósito de red:** Confirmar que el pod tiene egress hacia internet (sin NetworkPolicy restrictiva).

## 0. Verificación del Entorno y Librerías

In [ ]:
import sys, os, platform, psutil, time

print("="*65)
print("ENTORNO DE EJECUCIÓN — PERFIL ESTUDIANTE")
print("="*65)
print(f"  Python:         {sys.version.split()[0]}")
print(f"  Pod hostname:   {platform.node()}")
print(f"  CPUs lógicas:   {os.cpu_count()}")
mem = psutil.virtual_memory()
print(f"  RAM total:      {mem.total/1024**3:.2f} GB (límite K8s: 4 GB)")
print(f"  RAM en uso:     {mem.used/1024**3:.2f} GB  ({mem.percent:.1f}%)")
print("="*65)

# Verificar paquetes necesarios
required = ["pandas", "numpy", "matplotlib", "seaborn",
            "sklearn", "scipy", "statsmodels", "requests"]
missing = []
for pkg in required:
    try:
        __import__(pkg)
        print(f"  ✅ {pkg}")
    except ImportError:
        print(f"  ❌ {pkg} — no instalado")
        missing.append(pkg)

if missing:
    print(f"\n⚠️  Instalar con: pip install {' '.join(missing)}")

## 1. Descarga del Dataset desde Internet

Usamos el dataset **Water Quality** de Kaggle/UCI.  
Variables: `ph`, `Hardness`, `Solids`, `Chloramines`, `Sulfate`, `Conductivity`, `Organic_carbon`, `Trihalomethanes`, `Turbidity`, `Potability` (binario).

> La descarga valida que el pod tiene **conectividad de red egress** — un requisito para que los estudiantes puedan acceder a APIs externas.

In [ ]:
import requests
import io
import pandas as pd
import numpy as np

# Dataset Water Quality — GitHub raw (fuente abierta, sin auth)
DATASET_URL = (
    "https://raw.githubusercontent.com/dsrscientist/dataset1/"
    "master/water_potability.csv"
)

FALLBACK_URL = (
    "https://raw.githubusercontent.com/MainakRepositor/Datasets/"
    "master/Water%20Quality.csv"
)

print("🌐 Intentando descarga del dataset Water Quality...")
t_start = time.time()

df = None
for url in [DATASET_URL, FALLBACK_URL]:
    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        df = pd.read_csv(io.StringIO(response.text))
        elapsed = time.time() - t_start
        size_kb = len(response.content) / 1024
        print(f"✅ Descarga exitosa desde: {url}")
        print(f"   Tamaño: {size_kb:.1f} KB | Tiempo: {elapsed:.2f}s")
        print(f"   Velocidad efectiva: {size_kb/elapsed:.1f} KB/s")
        break
    except Exception as e:
        print(f"⚠️  Falló {url}: {e}")

if df is None:
    print("\n⚠️  Sin conectividad o URLs caídas. Generando dataset sintético...")
    np.random.seed(0)
    n = 3276
    df = pd.DataFrame({
        "ph":               np.random.uniform(0, 14, n),
        "Hardness":         np.random.normal(196, 32, n),
        "Solids":           np.random.normal(22014, 8768, n),
        "Chloramines":      np.random.normal(7.1, 1.6, n),
        "Sulfate":          np.random.normal(333, 41, n),
        "Conductivity":     np.random.normal(426, 81, n),
        "Organic_carbon":   np.random.normal(14, 3.3, n),
        "Trihalomethanes":  np.random.normal(66, 16, n),
        "Turbidity":        np.random.normal(3.97, 0.78, n),
        "Potability":       np.random.randint(0, 2, n),
    })
    df.loc[np.random.choice(n, 500, replace=False), "ph"] = np.nan
    df.loc[np.random.choice(n, 400, replace=False), "Sulfate"] = np.nan
    print(f"✅ Dataset sintético generado: {len(df):,} filas")

print(f"\n  Filas: {len(df):,} | Columnas: {df.columns.tolist()}")

## 2. EDA Rápido del Dataset de Calidad de Agua

In [ ]:
print("--- RESUMEN ESTADÍSTICO ---")
print(df.describe().round(3).to_string())
print(f"\n--- VALORES NULOS ---")
print(df.isnull().sum().to_string())
print(f"\n--- DISTRIBUCIÓN DE POTABILIDAD ---")
print(df["Potability"].value_counts().rename({0: "No potable", 1: "Potable"}).to_string())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Imputar nulos con la mediana (preprocesamiento básico)
df_clean = df.copy()
for col in df_clean.columns:
    df_clean[col].fillna(df_clean[col].median(), inplace=True)

sns.set_theme(style="whitegrid")
features = ["ph", "Hardness", "Chloramines", "Conductivity",
            "Turbidity", "Organic_carbon"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("EDA — Calidad del Agua (Perfil Estudiante)", fontsize=13, fontweight="bold")

for ax, col in zip(axes.flatten(), features):
    sns.histplot(data=df_clean, x=col, hue="Potability",
                 bins=40, kde=True, ax=ax, palette={0: "salmon", 1: "steelblue"},
                 legend=(col == "ph"))
    ax.set_title(col)
    ax.set_xlabel("")

plt.tight_layout()
plt.savefig("/tmp/nb2_eda_agua.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ EDA completado.")

## 3. Modelo 1: Regresión Lineal Múltiple

**Target:** `ph` (continuo).  
**Predictores:** `Hardness`, `Chloramines`, `Sulfate`, `Conductivity`, `Organic_carbon`, `Trihalomethanes`.

Se mide el **tiempo de entrenamiento** para validar el rendimiento dentro del límite de 2 CPU.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
import time

features_lr = ["Hardness", "Chloramines", "Sulfate",
               "Conductivity", "Organic_carbon", "Trihalomethanes"]
target_lr   = "ph"

X = df_clean[features_lr].values
y = df_clean[target_lr].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# ──────────────────────────────────────────────
print("⏱️  Iniciando entrenamiento: Regresión Lineal Múltiple...")
t0 = time.perf_counter()

model_lr = LinearRegression()
model_lr.fit(X_train_s, y_train)

t_train = time.perf_counter() - t0
print(f"   Entrenamiento completado en: {t_train*1000:.2f} ms")
# ──────────────────────────────────────────────

y_pred = model_lr.predict(X_test_s)
r2   = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# Cross-validation
t0 = time.perf_counter()
cv_scores = cross_val_score(model_lr, X_train_s, y_train, cv=5, scoring="r2")
t_cv = time.perf_counter() - t0

print()
print("="*50)
print("  RESULTADOS — Regresión Lineal Múltiple")
print("="*50)
print(f"  R²  en test:         {r2:.4f}")
print(f"  RMSE en test:        {rmse:.4f}")
print(f"  R² CV (5-fold):      {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"  Tiempo CV:           {t_cv:.3f} s")
print()
print("  Coeficientes:")
for feat, coef in zip(features_lr, model_lr.coef_):
    print(f"    {feat:<30} {coef:+.4f}")
print(f"    Intercepto:                    {model_lr.intercept_:+.4f}")
print("="*50)

## 4. Modelo 2: ANOVA de Un Factor

**Pregunta:** ¿Difiere significativamente el nivel de **Turbidez** entre muestras de agua **potable vs no potable**?  
Usamos `scipy.stats.f_oneway` (ANOVA) y `statsmodels` para la tabla ANOVA completa.

In [ ]:
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
import warnings
warnings.filterwarnings("ignore")

potable     = df_clean.loc[df_clean["Potability"]==1, "Turbidity"].values
no_potable  = df_clean.loc[df_clean["Potability"]==0, "Turbidity"].values

print("⏱️  Ejecutando ANOVA de un factor...")
t0 = time.perf_counter()

# --- ANOVA scipy ---
f_stat, p_value = stats.f_oneway(potable, no_potable)
t_anova = time.perf_counter() - t0
print(f"   ANOVA scipy completado en: {t_anova*1000:.3f} ms")

# --- ANOVA statsmodels ---
t0 = time.perf_counter()
model_anova = ols("Turbidity ~ C(Potability)", data=df_clean).fit()
anova_table = sm.stats.anova_lm(model_anova, typ=1)
t_sm = time.perf_counter() - t0
print(f"   ANOVA statsmodels en:      {t_sm*1000:.3f} ms")

print()
print("="*60)
print("  RESULTADOS — ANOVA: Turbidez vs Potabilidad")
print("="*60)
print(f"  Grupos:   Potable n={len(potable):,} | No Potable n={len(no_potable):,}")
print(f"  Media Potable:    {potable.mean():.4f}")
print(f"  Media No Potable: {no_potable.mean():.4f}")
print(f"  F-statistic:      {f_stat:.6f}")
print(f"  p-value:          {p_value:.6f}")
alpha = 0.05
conclusion = "RECHAZAMOS H₀" if p_value < alpha else "NO rechazamos H₀"
print(f"  α = {alpha} → {conclusion} (diferencia {'significativa' if p_value < alpha else 'NO significativa'})")
print()
print("  Tabla ANOVA (statsmodels):")
print(anova_table.to_string())
print("="*60)

In [ ]:
# Visualización comparativa de los dos modelos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Resumen de Modelos Estadísticos (Perfil Estudiante)",
             fontsize=12, fontweight="bold")

# Panel 1: Residuos de la regresión
residuos = y_test - y_pred
axes[0].scatter(y_pred, residuos, alpha=0.4, s=15, color="steelblue")
axes[0].axhline(0, color="red", linewidth=1.5, linestyle="--")
axes[0].set_xlabel("Valores Predichos")
axes[0].set_ylabel("Residuos")
axes[0].set_title(f"Regresión Lineal — Residuos vs Predichos\nR² = {r2:.4f}, RMSE = {rmse:.4f}")

# Panel 2: ANOVA — boxplot turbidez
plot_data = df_clean[["Turbidity", "Potability"]].copy()
plot_data["Clase"] = plot_data["Potability"].map({0: "No potable", 1: "Potable"})
sns.boxplot(data=plot_data, x="Clase", y="Turbidity",
            palette={"No potable": "salmon", "Potable": "steelblue"},
            ax=axes[1], width=0.4)
axes[1].set_title(f"ANOVA — Turbidez por Clase\nF={f_stat:.4f}, p={p_value:.4f}")
axes[1].set_xlabel("")
axes[1].set_ylabel("Turbidez (NTU)")

plt.tight_layout()
plt.savefig("/tmp/nb2_modelos.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ Visualización de modelos generada.")

---
## Resumen del Notebook 2

| Validación | Resultado |
|---|---|
| Conectividad de red egress del pod | ✅ Dataset descargado |
| EDA dentro de límite 4GB RAM | ✅ Exitoso |
| Regresión Lineal Múltiple (sklearn) | ✅ Tiempo < 1s con 2 CPU |
| ANOVA de un factor (scipy/statsmodels) | ✅ Tiempo < 100ms |
| Límites CPU/RAM respetados (cgroups) | ✅ Sin OOM Kill |

> La calidad predictiva baja (R² ~ 0.05) es **esperada** — el pH del agua no es linealmente predecible
> desde estas variables. El objetivo del notebook es **validar la infraestructura**, no construir el mejor modelo.